In [29]:
import re
import calendar
from pathlib import Path

import pandas as pd
import pdfplumber

In [30]:
PROJECT_ROOT = Path.cwd()

if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw_pdfs"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"

print("Raw PDF folder:", RAW_DIR)
print("PDF count:", len(list(RAW_DIR.rglob("*.pdf"))))

Raw PDF folder: /Users/serratak/Desktop/visa-bulletin-project/visa-bulletin-project/data/raw_pdfs
PDF count: 120


In [31]:
AREAS = [
    "All Chargeability Areas Except Those Listed",
    "China",
    "India",
    "Mexico",
    "Philippines"
]

ROW_PATTERN = re.compile(
    r"^(F1|F2A|F2B|F3|F4)\s+"
    r"((?:\d{2}[A-Z]{3}\d{2}|C|U))\s+"
    r"((?:\d{2}[A-Z]{3}\d{2}|C|U))\s+"
    r"((?:\d{2}[A-Z]{3}\d{2}|C|U))\s+"
    r"((?:\d{2}[A-Z]{3}\d{2}|C|U))\s+"
    r"((?:\d{2}[A-Z]{3}\d{2}|C|U))\s*$"
)

In [32]:
MONTHS = {
    month.lower(): number
    for number, month in enumerate(calendar.month_name)
    if month
}

def get_bulletin_date(pdf_path):
    month_name, year = pdf_path.stem.rsplit("_", 1)

    month_number = MONTHS[month_name.lower()]

    date = pd.Timestamp(
        year=int(year),
        month=month_number,
        day=1
    )

    return date + pd.offsets.MonthEnd(0)

In [33]:
#test
get_bulletin_date(
    RAW_DIR / "2016" / "october_2016.pdf"
)

Timestamp('2016-10-31 00:00:00')

In [43]:
def parse_old_format(pdf_path):

    with pdfplumber.open(pdf_path) as pdf:
        text = "\n".join(
            page.extract_text() or ""
            for page in pdf.pages
        )

    upper_text = text.upper()

    final_start = upper_text.find(
        "FINAL ACTION DATES FOR FAMILY-SPONSORED"
    )

    filing_start = upper_text.find(
        "DATES FOR FILING FAMILY-SPONSORED"
    )

    if final_start == -1 or filing_start == -1:
        return []

    final_section = text[final_start:filing_start]
    filing_section = text[filing_start:]

    bulletin_date = get_bulletin_date(pdf_path)

    records = []

    for table_type, section in [
        ("Final Action", final_section),
        ("Dates for Filing", filing_section)
    ]:

        for line in section.splitlines():
            parts = line.strip().split()

            if len(parts) == 6 and parts[0] in {
                "F1", "F2A", "F2B", "F3", "F4"
            }:
                category = parts[0]

                values = [
                    normalize_cutoff_value(value)
                    for value in parts[1:]
                ]

                if all(DATE_PATTERN.match(value) for value in values):

                    for area, value in zip(AREAS, values):
                        records.append({
                            "bulletin_date": bulletin_date,
                            "visa_category": category,
                            "chargeability_area": area,
                            "table_type": table_type,
                            "raw_cutoff_value": value
                        })

    return records

In [44]:
def normalize_cutoff_value(value):
    value = value.strip().upper()

    if value in {"C", "U"}:
        return value

    if len(value) == 7:
        chars = list(value)

        for i in [0, 1, 5, 6]:
            if chars[i] == "O":
                chars[i] = "0"

        value = "".join(chars)

    return value

In [45]:
test_records = parse_old_format(
    RAW_DIR / "2016" / "october_2016.pdf"
)

print("Records:", len(test_records))

pd.DataFrame(test_records).head(50)

Records: 50


,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value
0,2016-10-31,F1,All Chargeability Areas Except Those Listed,Final Action,22SEP09
1,2016-10-31,F1,China,Final Action,22SEP09
2,2016-10-31,F1,India,Final Action,22SEP09
3,2016-10-31,F1,Mexico,Final Action,01APR95
4,2016-10-31,F1,Philippines,Final Action,01AUG05
5,2016-10-31,F2A,All Chargeability Areas Except Those Listed,Final Action,22DEC14
6,2016-10-31,F2A,China,Final Action,22DEC14
7,2016-10-31,F2A,India,Final Action,22DEC14
8,2016-10-31,F2A,Mexico,Final Action,01DEC14
9,2016-10-31,F2A,Philippines,Final Action,22DEC14


In [46]:
old_format_files = []

for pdf_path in sorted(RAW_DIR.rglob("*.pdf")):
    bulletin_date = get_bulletin_date(pdf_path)

    if bulletin_date <= pd.Timestamp("2025-10-31"):
        old_format_files.append(pdf_path)

print("Old-format PDF count:", len(old_format_files))

Old-format PDF count: 109


In [47]:
all_old_records = []
validation_results = []

for pdf_path in old_format_files:
    records = parse_old_format(pdf_path)

    validation_results.append({
        "file": pdf_path.name,
        "records": len(records)
    })

    all_old_records.extend(records)

In [48]:
validation_df = pd.DataFrame(validation_results)

validation_df[validation_df["records"] != 50]

,file,records


In [49]:
len(all_old_records)

5450

In [50]:
old_df = pd.DataFrame(all_old_records)

old_df.shape

(5450, 5)

In [54]:
print(all_old_records)
pd.DataFrame(all_old_records)

[{'bulletin_date': Timestamp('2016-12-31 00:00:00'), 'visa_category': 'F1', 'chargeability_area': 'All Chargeability Areas Except Those Listed', 'table_type': 'Final Action', 'raw_cutoff_value': '01DEC09'}, {'bulletin_date': Timestamp('2016-12-31 00:00:00'), 'visa_category': 'F1', 'chargeability_area': 'China', 'table_type': 'Final Action', 'raw_cutoff_value': '01DEC09'}, {'bulletin_date': Timestamp('2016-12-31 00:00:00'), 'visa_category': 'F1', 'chargeability_area': 'India', 'table_type': 'Final Action', 'raw_cutoff_value': '01DEC09'}, {'bulletin_date': Timestamp('2016-12-31 00:00:00'), 'visa_category': 'F1', 'chargeability_area': 'Mexico', 'table_type': 'Final Action', 'raw_cutoff_value': '15APR95'}, {'bulletin_date': Timestamp('2016-12-31 00:00:00'), 'visa_category': 'F1', 'chargeability_area': 'Philippines', 'table_type': 'Final Action', 'raw_cutoff_value': '15SEP05'}, {'bulletin_date': Timestamp('2016-12-31 00:00:00'), 'visa_category': 'F2A', 'chargeability_area': 'All Chargeabili

,bulletin_date,visa_category,chargeability_area,table_type,raw_cutoff_value
0,2016-12-31,F1,All Chargeability Areas Except Those Listed,Final Action,01DEC09
1,2016-12-31,F1,China,Final Action,01DEC09
2,2016-12-31,F1,India,Final Action,01DEC09
3,2016-12-31,F1,Mexico,Final Action,15APR95
4,2016-12-31,F1,Philippines,Final Action,15SEP05
...,...,...,...,...,...
5445,2025-09-30,F4,All Chargeability Areas Except Those Listed,Dates for Filing,01JAN09
5446,2025-09-30,F4,China,Dates for Filing,01JAN09
5447,2025-09-30,F4,India,Dates for Filing,01DEC06
5448,2025-09-30,F4,Mexico,Dates for Filing,30APR01
